# Persona C — Clasificador, N-gramas y Sentimiento

Notebook individual para probar `classifier.py`, `ngram_model.py` y `sentiment.py`
contra el dataset real, **sin tocar `main.py`**. Al final del taller, una sola
persona integra todo en `main.py`.

**Flujo de trabajo:**
1. Clonar el repo (o `git pull` si ya lo clonaste antes).
2. Instalar dependencias.
3. Cargar el dataset real con `cargar_dataset()`.
4. Probar tu parte del pipeline de forma aislada.
5. Cuando todo funcione, avisas al equipo y se integra a `main.py`.

## 1. Clonar el repositorio

Reemplaza la URL por la de su repo de GitHub. Esto descarga el proyecto completo
(incluyendo `src/`) a la máquina virtual de Colab.

In [ ]:
!git clone https://github.com/USUARIO/biblical_text_mining.git
%cd biblical_text_mining

Si ya lo habías clonado en una sesión anterior de Colab y solo quieres traer
los cambios más recientes de tus compañeros, usa esto en vez del clone:

```python
%cd /content/biblical_text_mining
!git pull
```

## 2. Instalar dependencias

In [ ]:
!pip install -q -r requirements.txt

## 3. Dataset

El dataset de Kaggle (`oswinrh/bible`) pesa pocos MB, así que **lo más simple es
subirlo directo al repo** dentro de `data/` y listo (cada persona lo tiene al hacer
`git clone`/`git pull`).

Si prefieren no subirlo a GitHub, otra opción es subir los 3 CSV manualmente en
cada sesión de Colab con el panel de archivos de la izquierda (ícono de carpeta),
arrastrándolos a `biblical_text_mining/data/`.

In [ ]:
from pathlib import Path
from src.data_loader import cargar_dataset
from src.models import Biblia

dir_dataset = Path("data")
path_bible = dir_dataset / "t_asv.csv"
path_key = dir_dataset / "key_english.csv"
path_genre = dir_dataset / "key_genre_english.csv"

df_raw = cargar_dataset(path_bible, path_key, path_genre)
df_raw.head()

In [ ]:
biblia = Biblia.from_dataframe(
    df_raw,
    col_libro="Book Name",
    col_testamento="Testament (OT or NT)",
    col_capitulo="Chapter",
    col_versiculo="Verse",
    col_texto="Text",
    col_genero="Genre name",
)
print(biblia)
print(biblia.get_resumen())
print(biblia.get_resumen_generos())

df = biblia.to_dataframe()
df.head()

## 4. Preprocesamiento y TF-IDF

Necesitas esto como insumo para el clasificador (features) y para el generador
de n-gramas (texto tokenizado). Esta parte la hace Persona A / B, pero la
ejecutas aquí también para poder probar tu propio módulo de forma aislada.

In [ ]:
from src.preprocessing import TextPreprocessor
from src.tfidf import TFIDFVectorizer

preprocessor = TextPreprocessor()
df["texto_procesado"] = preprocessor.process_corpus(df["texto_original"].tolist())

vectorizer = TFIDFVectorizer()
matriz_tfidf = vectorizer.fit_transform(df["texto_procesado"].tolist())
matriz_tfidf.shape

## 5. Clasificador de versículos (`classifier.py`)

In [ ]:
from src.classifier import VerseClassifier

clasificador = VerseClassifier(modelo="logistic")
clasificador.entrenar(matriz_tfidf, df["libro"])
resultados = clasificador.evaluar()

print(clasificador)
print("Accuracy:", resultados["accuracy"])
print(resultados["reporte"])

In [ ]:
# Matriz de confusión (usa visualization.py si ya está disponible)
from src import visualization as viz

viz.plot_matriz_confusion(resultados["matriz_confusion"], resultados["clases"])

## 6. Generador de texto con n-gramas (`ngram_model.py`)

In [ ]:
from src.ngram_model import NGramModel, comparar_modelos

resultados_ngram = comparar_modelos(df["texto_procesado"].tolist(), ns=(1, 2, 3, 4), max_len=20)
for n, texto in resultados_ngram.items():
    print(f"n={n}: {texto}")

In [ ]:
# Probar un modelo individual, empezando desde una palabra específica
trigram = NGramModel(3).fit(df["texto_procesado"].tolist())
print(trigram)
print(trigram.generar(palabra_inicial="dios", max_len=20))

## 7. Análisis de sentimiento (`sentiment.py`)

In [ ]:
from src.sentiment import LexiconSentimentAnalyzer, calcular_sentimiento_corpus, agregar_por_libro, agregar_por_capitulo

analizador = LexiconSentimentAnalyzer()
df = calcular_sentimiento_corpus(df, analizador)

sentimiento_libro = agregar_por_libro(df)
sentimiento_capitulo = agregar_por_capitulo(df)

sentimiento_libro.head(10)

In [ ]:
viz.plot_sentimiento_por_libro(sentimiento_libro)

## 8. Notas para subir tus cambios

Si modificaste algún archivo de `src/` (ej. ajustaste el lexicón de
`sentiment.py`), desde Colab puedes hacer commit y push directo:

```python
!git config --global user.email "tu_correo@ucn.cl"
!git config --global user.name "Tu Nombre"
!git add src/sentiment.py notebooks/persona_c.ipynb
!git commit -m "Ajusta lexicon de sentimiento y agrega pruebas"
!git push
```

Para que el `push` funcione necesitas autenticarte (ej. con un Personal Access
Token de GitHub en vez de tu contraseña). Si prefieren algo más simple,
también pueden simplemente descargar este notebook (`Archivo > Descargar > .ipynb`)
y subirlo a GitHub manualmente desde la web.

**Importante:** como cada persona trabaja en su propio notebook
(`persona_a.ipynb`, `persona_b.ipynb`, `persona_c.ipynb`), no debería haber
conflictos de Git entre ustedes. El único momento de integración real es
cuando alguien copie las piezas ya probadas a `main.py`.